# Construction d'un Index Vectoriel avec FAISS et Sentence-Transformers

**Objectif :** Créer une base de données vectorielle performante pour l'interrogation de documents hétérogènes (PDF, TXT, JSON).

Ce notebook couvre les étapes de nettoyage (Preprocessing), de découpage (Chunking), d'encodage (Embedding) et de recherche de similarité.

---

# Partie 1 : Ingestion et Préparation des Données

Cette première partie du projet se concentre sur la création d'une base de connaissance propre à partir de documents bruts.

**Objectifs de cette section :**
1.  **Collecte** : Lire des fichiers hétérogènes (`.pdf`, `.txt`, `.json`) depuis le dossier `data/raw_docs`.
2.  **Nettoyage** : Standardiser le texte et retirer le bruit.
3.  **Chunking** : Découper les documents en segments de taille fixe avec chevauchement (overlap) pour ne pas perdre le contexte.
4.  **Sauvegarde** : Générer un fichier `docs_corpus.csv` structuré qui servira de base à notre moteur de recherche.

python src/docs_to_corpus.py

## 1. Configuration et Importations

In [5]:
# IMPORTS
from pathlib import Path
import sys
import re
import json
import pandas as pd

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

## 2. Configuration des Chemins et Structure

Nous définissons ici les chemins relatifs pour l'entrée et la sortie des données. Cela permet au notebook de fonctionner sur n'importe quelle machine tant que la structure du projet est respectée.

In [ ]:
from pathlib import Path

sys.path.append("..") 

# CONFIGURATION DES CHEMINS
BASE_DIR = Path("..") 

# Dossier d'entrée
RAW_DOC_DIR = BASE_DIR / "data" / "raw_docs"

# Dossier de sortie
PROC_DIR = BASE_DIR / "data" / "processed"

# On crée le dossier s'il n'existe pas
PROC_DIR.mkdir(parents=True, exist_ok=True)

# Fichier final
OUT_PATH = PROC_DIR / "docs_corpus.csv"

## 3. Fonctions d'Extraction (Readers)

Afin d'uniformiser le traitement, nous définissons des fonctions spécifiques pour extraire le texte brut de différents formats de fichiers.

Chaque fonction prend un `Path` en entrée et retourne une `str` (le contenu textuel).

### Détail des logiques d'extraction :

* **Fichiers TXT (`read_txt`)** : Lecture directe en `utf-8`.
* **Fichiers PDF (`read_pdf`)** :
    * Utilise la librairie `PyPDF2`.
    * Itere sur toutes les pages pour concaténer le texte.
    * Gère les erreurs de lecture (fichiers corrompus ou images sans OCR).
* **Fichiers JSON/JSONL (`read_json_as_text`)** :
    * Supporte le **JSON standard** (objets ou listes) et le **JSON Lines** (`.jsonl`).
    * **Stratégie heuristique** : Cherche prioritairement des clés explicites (`text`, `content`, `body`). Si aucune n'est trouvée, convertit l'objet entier en chaîne de caractères.

In [8]:
# FONCTIONS D'EXTRACTION (READERS)

def read_txt(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")


def read_pdf(path: Path) -> str:
    try:
        from PyPDF2 import PdfReader
    except ImportError:
        print("[ERROR] PyPDF2 is not installed. Please run: pip install PyPDF2")
        return ""

    text_parts: list[str] = []
    try:
        reader = PdfReader(str(path))
        for page in reader.pages:
            # extract_text() peut renvoyer None sur des pages vides/images
            page_text = page.extract_text() or ""
            text_parts.append(page_text)
    except Exception as e:
        print(f"[ERROR] Failed to read PDF {path}: {e}")
        return ""

    return "\n\n".join(text_parts)


def read_json_as_text(path: Path) -> str:
    texts: list[str] = []

    def _extract_from_obj(obj):
        if isinstance(obj, str):
            texts.append(obj)
        elif isinstance(obj, dict):
            # Stratégie heuristique : on cherche les clés communes contenant du texte
            for key in ("text", "content", "body"):
                if key in obj and isinstance(obj[key], str):
                    texts.append(obj[key])
                    return
            # Si aucune clé connue, on dump l'objet en JSON string
            texts.append(json.dumps(obj, ensure_ascii=False))
        else:
            texts.append(json.dumps(obj, ensure_ascii=False))

    try:
        # Cas 1 : JSON Lines (.jsonl) : Un objet JSON par ligne
        if path.suffix.lower() == ".jsonl":
            with path.open(encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line: continue
                    try:
                        obj = json.loads(line)
                    except json.JSONDecodeError:
                        continue
                    _extract_from_obj(obj)
        # Cas 2 : JSON Standard (.json) : Liste ou Objet unique
        else:
            with path.open(encoding="utf-8") as f:
                data = json.load(f)
            if isinstance(data, list):
                for item in data:
                    _extract_from_obj(item)
            else:
                _extract_from_obj(data)
    except Exception as e:
        print(f"[ERROR] Failed to read JSON {path}: {e}")
        return ""

    return "\n\n".join(texts)


## 4. Nettoyage et Découpage (Chunking)

Une fois le texte brut extrait, il doit être préparé pour l'embedding.

Nous utilisons deux fonctions distinctes :

### 1. Nettoyage Basique (`basic_clean`)
Cette fonction normalise le texte avant traitement :
* **Normalisation** : Conversion des sauts de ligne Windows/Mac en format standard (`\n`).
* **Segmentation** : Découpage en paragraphes basés sur les lignes vides.
* **Filtrage** : Suppression du "bruit" (headers, numéros de page, segments < 20 caractères).

### 2. Découpage Intelligent (`optimized_chunk_text`)


Contrairement à un découpage brutal tous les X caractères, cette fonction respecte la grammaire et le contexte.

**Algorithme :**
1.  **Respect des phrases** : Le texte est d'abord divisé phrase par phrase (via regex `.!?`). On ne coupe jamais au milieu d'une phrase (sauf cas extrême).
2.  **Fenêtre glissante (Sliding Window)** : On remplit un "chunk" jusqu'à atteindre `max_words` (par défaut initialisé à 300 mots).
3.  **Gestion du Chevauchement (Overlap)** :
    * Une fois un chunk plein, on ne repart pas de zéro.
    * On effectue un **backtracking** : on récupère les dernières phrases du chunk précédent (jusqu'à `overlap_words`, par défaut initialisé à 50 mots).
    * Cela garantit qu'aucune information n'est perdue à la "frontière" de deux morceaux de texte.

In [9]:
# FONCTIONS DE NETTOYAGE ET DÉCOUPAGE (PRE-PROCESSING)

def basic_clean(text: str) -> list[str]:
    # Normalisation des retours à la ligne
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Séparation en paragraphes en utilisant les lignes vides comme séparateurs
    raw_paragraphs = re.split(r"\n\s*\n", text)

    paragraphs: list[str] = []
    for para in raw_paragraphs:
        # Fusionner les espaces vides en 1
        para = re.sub(r"\s+", " ", para).strip()
        
        # Filtre de qualité : On ignore les headers, numéros de page, ou bruit < 20 caractères
        if len(para) < 20:
            continue
        paragraphs.append(para)

    return paragraphs

import re
from typing import List

def optimized_chunk_text(
    paragraphs: List[str], 
    max_words: int = 300, 
    overlap_words: int = 50
) -> List[str]:
    """
    Découpe le texte en respectant les phrases et en ajoutant un chevauchement.
    
    Args:
        paragraphs: Liste des paragraphes bruts.
        max_words: Nombre maximum de mots par chunk.
        overlap_words: Nombre de mots à reprendre du chunk précédent (chevauchement).
    """
    
    # 1. Nettoyage et unification du texte
    # On joint par \n\n pour garder la trace des paragraphes si besoin, 
    # mais on traite le texte globalement pour le découpage.
    full_text = "\n\n".join(p.strip() for p in paragraphs if p.strip())
    
    # 2. Découpage intelligent en phrases
    # Regex : Coupe sur (. ! ?) suivi d'un espace ou d'une fin de ligne.
    # Le pattern inclut le délimiteur dans le résultat pour ne pas perdre la ponctuation.
    sentence_endings = re.compile(r'(?<=[.!?])\s+')
    sentences = sentence_endings.split(full_text)
    
    chunks: List[str] = []
    current_chunk_sentences: List[str] = []
    current_word_count = 0
    
    i = 0
    while i < len(sentences):
        sentence = sentences[i].strip()
        if not sentence:
            i += 1
            continue
            
        sentence_word_count = len(sentence.split())

        # CAS A : La phrase seule est plus grande que la limite (Edge Case)
        # On doit la couper brutalement (fallback sur logique mots)
        if sentence_word_count > max_words:
            # Si on a du contenu en attente, on le sauvegarde d'abord
            if current_chunk_sentences:
                chunks.append(" ".join(current_chunk_sentences))
                current_chunk_sentences = []
                current_word_count = 0
            
            # Découpage récursif de la phrase géante
            words = sentence.split()
            for j in range(0, len(words), max_words - overlap_words):
                # On crée des sous-chunks avec overlap forcé
                sub_chunk = " ".join(words[j : j + max_words])
                chunks.append(sub_chunk)
            
            i += 1
            continue

        # CAS B : Ajout possible dans le chunk courant
        if current_word_count + sentence_word_count <= max_words:
            current_chunk_sentences.append(sentence)
            current_word_count += sentence_word_count
            i += 1
        
        # CAS C : Le chunk est plein
        else:
            # 1. Sauvegarder le chunk actuel
            chunks.append(" ".join(current_chunk_sentences))
            
            # 2. Gérer l'overlap (Backtracking)
            # On recule dans les phrases du chunk actuel pour récupérer ~overlap_words
            overlap_buffer = []
            overlap_count = 0
            
            # On parcourt à l'envers le chunk qu'on vient de fermer
            for prev_sent in reversed(current_chunk_sentences):
                prev_len = len(prev_sent.split())
                if overlap_count + prev_len <= overlap_words:
                    overlap_buffer.insert(0, prev_sent) # On remet au début
                    overlap_count += prev_len
                else:
                    # On arrête dès qu'on a assez de contexte
                    break
            
            # 3. Réinitialiser le chunk courant avec le buffer d'overlap
            current_chunk_sentences = overlap_buffer[:]
            current_word_count = overlap_count
            
            # Note : On n'incrémente pas 'i' ici, car la phrase actuelle (qui a déclenché
            # le dépassement) doit être testée à nouveau dans le nouveau chunk.

    # Ne pas oublier le dernier morceau
    if current_chunk_sentences:
        chunks.append(" ".join(current_chunk_sentences))

    return chunks

## 5. Pipeline d'Exécution

La fonction `docs_to_corpus` est le point d'entrée principal du script. Elle orchestre la transformation des documents bruts en un dataset structuré.



**Le flux de traitement est le suivant :**

1.  **Scan et Dispatch** : Parcourt le dossier `raw_docs` et choisit la bonne fonction de lecture selon l'extension (`.pdf`, `.txt`, `.json`).
2.  **Nettoyage** : Applique le filtre de qualité (suppression des en-têtes/bruit).
3.  **Chunking** : Découpe le texte en morceaux de ~300 mots avec chevauchement.
4.  **Structuration** : Associe chaque morceau à ses métadonnées (Source, ID).
5.  **Sauvegarde** : Exporte le résultat final sous forme de fichier CSV.

### Schéma des données de sortie (`docs_corpus.csv`)
Le fichier généré contiendra les colonnes suivantes :

| Colonne | Type | Description |
| :--- | :--- | :--- |
| `doc_id` | `int` | Identifiant unique du document original (1, 2, 3...) |
| `chunk_id` | `int` | Ordre séquentiel du morceau dans le document (pour reconstruire le contexte si besoin) |
| `text` | `str` | Le contenu textuel du chunk (ce qui sera vectorisé) |
| `source` | `str` | Le nom du fichier d'origine (ex: `rapport_2023.pdf`) |

In [10]:
# docs_to_corpus()

def docs_to_corpus():
    """
    Point d'entrée du script d'ingestion.
    Parcourt RAW_DOC_DIR -> Extrait -> Nettoie -> Chunk -> Sauvegarde CSV.
    """
    rows = []
    doc_id = 1

    # Vérification dossier
    if not RAW_DOC_DIR.exists():
        print(f"[ERROR] RAW_DOC_DIR does not exist: {RAW_DOC_DIR}")
        print("Please create the folder and put your PDF/TXT files inside.")
        return

    print(f"[INFO] Scanning files in {RAW_DOC_DIR}...")

    # Boucle sur les fichiers
    for path in RAW_DOC_DIR.iterdir():
        if not path.is_file():
            continue

        suffix = path.suffix.lower()

        # 1. Extraction selon le type
        if suffix == ".txt":
            raw_text = read_txt(path)
        elif suffix == ".pdf":
            raw_text = read_pdf(path)
        elif suffix in {".json", ".jsonl"}:
            raw_text = read_json_as_text(path)
        else:
            print(f"[WARN] Skipping unsupported file type: {path.name}")
            continue

        # 2. Nettoyage
        paragraphs = basic_clean(raw_text)
        if not paragraphs:
            print(f"[WARN] Empty document after cleaning: {path.name}")
            continue

        # 3. Chunking (Découpage)
        chunks = optimized_chunk_text(paragraphs, max_words=300)
        if not chunks:
            print(f"[WARN] No chunks produced for: {path.name}")
            continue

        # 4. Structuration des données
        for chunk_id, chunk in enumerate(chunks, start=1):
            rows.append(
                {
                    "doc_id": doc_id,       # ID unique du document
                    "chunk_id": chunk_id,   # ID du morceau dans le doc
                    "text": chunk,          # Contenu
                    "source": path.name,    # Méta-donnée source
                }
            )

        print(f"[INFO] Processed {path.name}: {len(chunks)} chunks")
        doc_id += 1

    # 5. Sauvegarde Finale
    if not rows:
        print("[ERROR] No chunks generated. Check documents in data/raw.")
        return

    df = pd.DataFrame(rows)
    df.to_csv(OUT_PATH, index=False, encoding="utf-8")
    print(f"\n[OK] Pipeline Terminé.")
    print(f"     Saved {len(df)} chunks from {doc_id-1} documents to {OUT_PATH}")


docs_to_corpus()

[INFO] Scanning files in ../data/raw_docs...
[INFO] Processed HIV-26-1329.pdf: 36 chunks
[INFO] Processed s13045-024-01640-8.pdf: 30 chunks
[INFO] Processed s12933-022-01516-6.pdf: 57 chunks
[INFO] Processed RESP-30-574.pdf: 41 chunks
[INFO] Processed s13023-024-03342-3.pdf: 43 chunks
[INFO] Processed s12913-021-06331-5.pdf: 19 chunks
[INFO] Processed s12889-024-20624-4.pdf: 50 chunks

[OK] Pipeline Terminé.
     Saved 276 chunks from 7 documents to ../data/processed/docs_corpus.csv


# Partie 2 : Vectorisation, Indexation et Recherche Sémantique

Cette seconde partie transforme notre corpus de texte nettoyé (le CSV) en un moteur de recherche fonctionnel.



**Objectifs de cette section :**

1.  **Vectorisation (Embedding)** : Convertir chaque morceau de texte en un vecteur numérique de 384 dimensions grâce au modèle `sentence-transformers/all-MiniLM-L6-v2`.
2.  **Indexation (FAISS)** : Stocker ces vecteurs dans une structure optimisée (`IndexFlatIP`) pour permettre une recherche de similarité ultra-rapide.
3.  **Persistance** : Sauvegarder l'index et les métadonnées sur le disque (`data/index/`) pour ne pas avoir à tout recalculer à chaque lancement.
4.  **Inférence (Recherche)** : Simuler une requête utilisateur, la vectoriser, et retrouver les documents les plus pertinents par similarité cosinus.

## 1. Configuration de l'environnement de recherche

Nous définissons les chemins pour cette seconde phase.

* **Entrée** : Nous allons lire le fichier `docs_corpus.csv` généré dans la partie précédente.
* **Sortie** : Nous créons un dossier dédié `data/index/` pour sauvegarder nos index vectoriels. Cela permettra de charger le moteur de recherche plus tard sans avoir à tout recalculer.

In [12]:

sys.path.append("..") 

# CONFIGURATION DES CHEMINS
BASE_DIR = Path("..")

# Dossier où se trouve le CSV (Données traitées)
PROC_DIR = BASE_DIR / "data" / "processed"

# Dossier où on va sauvegarder l'index
INDEX_DIR = BASE_DIR / "data" / "index"

# Création du dossier s'il n'existe pas
INDEX_DIR.mkdir(parents=True, exist_ok=True)

## 2. Construction de l'Index Vectoriel (FAISS)

Cette fonction convertit notre texte en vecteurs mathématiques (embeddings) et crée l'index de recherche.



**Etapes clés :**

1.  **Modèle d'Embedding** : Nous utilisons `sentence-transformers/all-MiniLM-L6-v2`.
    * C'est un modèle léger et rapide, idéal pour faire tourner sur un CPU.
    * Il transforme chaque chunk en un vecteur de **384 dimensions**.
2.  **Normalisation** : Nous activons `normalize_embeddings=True`.
    * Cela force la longueur de tous les vecteurs à 1.
3.  **Index FAISS** : Nous utilisons `IndexFlatIP` (Inner Product).

**Sauvegarde des Artefacts (`data/index/`) :**
Pour éviter de tout recalculer à chaque fois, nous sauvegardons 4 fichiers :
* `corpus.index` : La structure de données FAISS optimisée pour la recherche.
* `corpus_embeddings.npy` : Les vecteurs bruts (numpy array) pour usage futur/debug.
* `corpus_meta.csv` : Le lien entre l'ID du vecteur et le texte original.
* `embedding_model.txt` : Le nom du modèle utilisé (pour garantir la compatibilité lors de la recherche).

In [13]:
def build_index():
    corpus_path = PROC_DIR / "docs_corpus.csv"
    df = pd.read_csv(corpus_path)

    print(f"Loaded {len(df)} document chunks for the knowledge base")

    texts = df["text"].astype(str).tolist()
    doc_ids = df["doc_id"].tolist()

    # 1. Chargement du modèle de sentence embedding
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    print(f"Loaded embedding model: {model_name}")
    model = SentenceTransformer(model_name)

    # 2. Encoder tous les documents
    print("Start encoding documents...")
    embeddings = model.encode(
        texts,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    print("Encoding finished, shape:", embeddings.shape)

    # 3. Construction de l'index FAISS
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    print("FAISS index built, number of vectors:", index.ntotal)

    # 4. Sauvegarde de l'index, des embeddings et des métadonnées des documents
    faiss_path = INDEX_DIR / "corpus.index"
    faiss.write_index(index, str(faiss_path))

    emb_path = INDEX_DIR / "corpus_embeddings.npy"
    np.save(emb_path, embeddings)

    meta_path = INDEX_DIR / "corpus_meta.csv"
    df.to_csv(meta_path, index=False, encoding="utf-8")

    # Sauvagardes du nom du modèle d'embedding pour pouvoir le charger plus tard
    model_name_path = INDEX_DIR / "embedding_model.txt"
    model_name_path.write_text(model_name, encoding="utf-8")

    print(f"Index saved to: {faiss_path}")
    print(f"Embeddings saved to: {emb_path}")
    print(f"Document metadata saved to: {meta_path}")
    print(f"Embedding model name saved to: {model_name_path}")

build_index()

Loaded 276 document chunks for the knowledge base
Loaded embedding model: sentence-transformers/all-MiniLM-L6-v2
Start encoding documents...


Batches: 100%|██████████| 5/5 [00:00<00:00, 10.74it/s]


Encoding finished, shape: (276, 384)
FAISS index built, number of vectors: 276
Index saved to: ../data/index/corpus.index
Embeddings saved to: ../data/index/corpus_embeddings.npy
Document metadata saved to: ../data/index/corpus_meta.csv
Embedding model name saved to: ../data/index/embedding_model.txt


In [15]:
# IMPORTS
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import sys


In [17]:

BASE_DIR = Path.cwd()
while not (BASE_DIR / "data").exists():
    if BASE_DIR == BASE_DIR.parent:
        raise FileNotFoundError("Dossier 'data' introuvable. Êtes-vous bien dans le projet ?")
    BASE_DIR = BASE_DIR.parent


if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))


PROC_DIR = BASE_DIR / "data" / "processed"
INDEX_DIR = BASE_DIR / "data" / "index"


INDEX_DIR.mkdir(parents=True, exist_ok=True)

## 3. Chargement des Ressources et Fonction de Recherche

Cette section met en place l'interface de recherche proprement dite.

Nous définissons deux fonctions distinctes pour simuler une application réelle :

1.  **`load_resources()`** : Reconstitue l'état du système à partir des fichiers sauvegardés sur le disque.
2.  **`search()`** : Exécute la requête sémantique.



**Le mécanisme de "Lookup" :**
FAISS ne stocke que des vecteurs et des IDs. Il ne connaît pas le texte.
* L'index nous renvoie les **indices** des voisins les plus proches.
* Nous utilisons ces indices (`idx`) pour récupérer le texte original dans le DataFrame via `df.iloc[idx]`.

In [18]:
def load_resources():
    # 1. Chargement des métadonnées
    corpus_path = PROC_DIR / "docs_corpus.csv"
    df = pd.read_csv(corpus_path)

    # 2. Chargement de l'index FAISS
    faiss_path = INDEX_DIR / "corpus.index"
    index = faiss.read_index(str(faiss_path))

    # 3. Charger le modèle d'embedding
    model_name_path = INDEX_DIR / "embedding_model.txt"
    model_name = model_name_path.read_text(encoding="utf-8").strip()
    print(f"Loaded embedding model: {model_name}")
    model = SentenceTransformer(model_name)

    return df, index, model


def search(query, df, index, model, top_k=5):
    # 1. Encoder la question
    query_emb = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    # 2. Recherche dans FAISS
    scores, indices = index.search(query_emb, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        row = df.iloc[idx]
        text = str(row["text"])
        doc_id = row.get("doc_id", idx)

        results.append(
            {
                "doc_id": doc_id,
                "score": float(score),
                "text": text,
            }
        )
    return results

## 5. Démo Interactive

La fonction ci-dessous lance une boucle interactive dans le notebook.

**Comment l'utiliser :**
1.  Exécutez la cellule.
2.  Le script va charger le modèle et l'index (cela peut prendre quelques secondes).
3.  Une zone de saisie apparait : tapez votre question en langage naturel en anglais.
4.  Le système renvoie les 5 passages les plus pertinents avec leur **score de confiance**.
5.  Tapez `q` ou `quit` pour arrêter la démo.

> **Interprétation du Score :**
> * **> 0.6** : Très pertinent.
> * **0.3 - 0.6** : Pertinence moyenne (contexte connexe).
> * **< 0.3** : Faible pertinence (bruit ou sujet non couvert).

In [21]:
# TEST
def test():
    df, index, model = load_resources()

    print("=== Simple retrieval demo: enter a query and press Enter ===")
    print("Type q or quit to exit.")

    while True:
        query = input("\nPlease enter your query: ").strip()
        if query.lower() in {"q", "quit", "exit"}:
            print("Au revoir ~")
            break

        results = search(query, df, index, model, top_k=5)

        print("\nTop-5 retrieval results:")
        for i, r in enumerate(results, start=1):
            # Montre uniquement les 200 premiers caractères pour éviter les longues sorties
            preview = r["text"][:200].replace("\n", " ")
            print(f"[{i}] doc_id={r['doc_id']}  score={r['score']:.4f}")
            print(f"    {preview}...")


test()

Loaded embedding model: sentence-transformers/all-MiniLM-L6-v2
=== Simple retrieval demo: enter a query and press Enter ===
Type q or quit to exit.

Top-5 retrieval results:
[1] doc_id=1  score=0.5145
    Mortality and HIV infection Seven publications investigating the impact of free ARVs on mortality were identified (see Table S1). Most of the studies ( n=6) showed a reduction of mortality with free A...
[2] doc_id=1  score=0.5017
    Secondary outcomes (mor-tality and HIV transmission), which are less dependent on a single factor, such as free access to ARVs, are pre- sented in the supplemental material. The findings are cat-egori...
[3] doc_id=1  score=0.4891
    As we work towards global HIV targets, increasing ARV coverag e, particularly through free and accessible programmes, is crucial to ensure that morepeople use ARVs and achieve viral suppressions, whic...
[4] doc_id=1  score=0.4835
    2008; 86(7):559-567. doi: 10.2471/blt.07.044248 41. Fennell C, Escudero D, Zash R, et al. 